# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The dataset includes multiple tables as record sets:

- Table 1: Clinical features
- Table 2: Hormone levels & imaging
- Table 3: Genetic variants

Let's enumerate available record sets and their fields using their `@id`s.

In [ ]:
# Explore the record sets in the dataset
record_sets = dataset.record_sets

for record_set in record_sets:
    print(f"Record Set: {record_set['@id']} - {record_set.get('name', '<no name>')}")
    fields = record_set.get('fields', [])
    for field in fields:
        print(f"  Field: {field['@id']} ({field.get('name', '<no name>')})")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

Use the `@id` values for record sets and fields identified above.

In [ ]:
# Extract data from each record set
# Example RecordSet IDs (replace with actual IDs from previous cell if changed)
record_set_ids = [
    'https://api.app.sen.science/frontiers/7810165/table1',  # Table 1: Clinical Features
    'https://api.app.sen.science/frontiers/7810165/table2',  # Table 2: Hormone Levels & Imaging
    'https://api.app.sen.science/frontiers/7810165/table3',  # Table 3: Genetic Variants
]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for {record_set_id}: {df.shape[0]} rows, {df.shape[1]} columns")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(2), '\n')
    except Exception as e:
        print(f"Failed to load {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll analyze Table 2 (Hormone Levels & Imaging) and:
- Select the numeric field: `https://api.app.sen.science/frontiers/7810165/table2/serum_17ohprog_level`
- Filter for patients with 17-OHP > threshold
- Normalize the field
- Optionally group by genotype if available

In [ ]:
# Choose the record set for Table 2
record_set_id = 'https://api.app.sen.science/frontiers/7810165/table2'
df = dataframes.get(record_set_id, pd.DataFrame())

# Select the numeric field (use field @id)
numeric_field_id = 'https://api.app.sen.science/frontiers/7810165/table2/serum_17ohprog_level'
# If the DataFrame columns are the string representation of @id, adjust accordingly

threshold = 10  # ng/dL
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Optionally, group by another field (e.g., genotype)
    group_field_id = 'https://api.app.sen.science/frontiers/7810165/table3/genotype'
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Field {numeric_field_id} not found in {record_set_id} columns: {df.columns.tolist()}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Let's plot the distribution of the normalized 17-OHP levels for Table 2 filtered records.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

field_norm = f"{numeric_field_id}_normalized"

if numeric_field_id in filtered_df.columns and field_norm in filtered_df.columns:
    plt.figure(figsize=(6, 4))
    sns.histplot(filtered_df[field_norm], kde=True, bins=5)
    plt.title("Normalized 17-OHP Levels Distribution")
    plt.xlabel("Normalized 17-OHP Level")
    plt.ylabel("Count")
    plt.show()
else:
    print(f"Unable to plot: required fields not present in filtered_df.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset provides a structured clinical, laboratory, and genetic overview for a rare patient cohort.
- Outlier detection, normalization, and grouping operations can be performed by referencing fields and record sets via their `@id`.
- Visualizations reveal the distribution characteristics of key hormone markers.
- For further analysis, always consult the Croissant schema and field `@id` for accurate referencing.